# 📚 Python 栈与队列 — LeetCode 复习笔记

> **覆盖题目**：LC 20 · LC 155 · LC 225 · LC 150 · LC 739 · LC 853 · LC 84 · LC 239  
> **核心技巧**：基础栈应用 · 辅助栈 · 单调栈 · 单调队列 · 数据结构设计

| 编号 | 题名 | 题型 | 难度 |
|------|------|------|------|
| 20   | 有效的括号 | 基础栈：括号匹配 | 🟢 |
| 155  | 最小栈 | 辅助栈（双栈同步） | 🟡 |
| 225  | 用队列实现栈 | 数据结构设计 | 🟢 |
| 150  | 逆波兰表达式求值 | 栈求值（后缀表达式）| 🟡 |
| 739  | 每日温度 | **单调递减栈**（找下一个更大元素）| 🟡 |
| 853  | 车队 | 排序 + 栈/单次遍历 | 🟡 |
| 84   | 柱状图中最大的矩形 | **单调递增栈** | 🔴 |
| 239  | 滑动窗口最大值 | **单调递减双端队列** | 🔴 |

---

## 🗺️ 栈/队列题型分类速查

```
括号/匹配类             → 栈（遇左压栈、遇右弹栈比对）        (LC 20)
求值类（后缀表达式）     → 栈（遇数压栈、遇符号弹两个算）       (LC 150)
数据结构设计：双栈同步   → 辅助栈维护额外信息（如最小值）        (LC 155)
栈/队列互相模拟          → 数据结构设计                       (LC 225, LC 232)
找下一个更大/更小元素     → 单调栈                            (LC 739, LC 496)
求柱状图/能接雨水/区间最值 → 单调栈                           (LC 84, LC 42)
车队/股票收盘/落石问题    → 排序 + 栈/单次扫描                (LC 853)
滑动窗口的最值            → 单调队列（双端队列 deque）          (LC 239)
```

---

### 三种"单调结构"全景对比 ⭐

| 结构 | 数据类型 | 特性 | 典型题 | 时间 |
|------|--------|------|--------|------|
| **单调递增栈** | `list`（仅末尾操作）| 栈底→栈顶递增 | LC 84 柱状图最大矩形 | O(n) |
| **单调递减栈** | `list`（仅末尾操作）| 栈底→栈顶递减 | LC 739 每日温度 | O(n) |
| **单调队列** | `collections.deque`（双端）| 头尾都能弹 | LC 239 滑动窗口最大值 | O(n) |

> **判断口诀**：要求"下一个更大/更小"→ 单调栈；要求"窗口内最值"→ 单调队列。  
> **均摊 O(n) 的原因**：每个元素**最多入栈/队列一次、最多出一次**，所以总操作 ≤ 2n。

In [ ]:
from typing import List, Optional
from collections import deque
print("环境就绪 ✅")

---
## Part 1 — 基础栈应用（匹配 + 求值）

**核心范式**：遇到"开"压栈，遇到"闭"弹栈并配对处理。

```python
stack = []
for item in seq:
    if 是"开" / 操作数:
        stack.append(item)
    else:  # 是"闭" / 运算符
        if not stack or 配对失败(stack[-1], item):
            return False  # 或抛错
        左 = stack.pop()
        处理(左, item) → 可能再压回
```

### 1. LC 20 — 有效的括号

| 项目 | 内容 |
|------|------|
| **题型** | 基础栈：括号匹配 |
| **时间复杂度** | O(n) |
| **空间复杂度** | O(n)（最坏栈大小=输入长度的一半）|
| **数据范围** | 1 ≤ s.length ≤ 10⁴，s 只含 `()[]{}` 六种字符 |

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "**最近的**才能与当前匹配" / 嵌套结构 | 栈（LIFO 后进先出）|
| 字符串只含成对符号（括号、引号、HTML 标签）| 用哈希表存配对关系，遇右括号查左括号 |
| 长度为奇数时一定不合法 | 提前剪枝 `if len(s) % 2: return False` |

### 🧠 算法
```
pairs = {')':'(', ']':'[', '}':'{'}    # 右括号 → 对应左括号
stack = []
for ch in s:
    if ch in pairs:                    # 右括号：弹栈比对
        if not stack or stack[-1] != pairs[ch]:
            return False
        stack.pop()
    else:                              # 左括号：压栈
        stack.append(ch)
return not stack                       # 栈空才完全匹配
```

### 🔑 三个边界情况

| 情况 | 处理 |
|------|------|
| 遇到右括号但栈为空 | 返回 False（多余的右括号）|
| 栈顶左括号与当前右括号不匹配 | 返回 False（顺序错误）|
| 遍历完后栈不空 | 返回 False（多余的左括号未闭合）|

### 哈希表存"右→左"而非"左→右"的好处
- 遇到右括号时直接 `pairs[ch]` 拿对应左括号，**判断 `ch in pairs` 同时判断了"是不是右括号"**——一举两得。
- 反向存就要先判断是不是右括号，再查字典，多一步。

In [ ]:
class Solution_20:
    def isValid(self, s: str) -> bool:
        if len(s) % 2: return False             # 奇数长度直接淘汰
        pairs = {')':'(', ']':'[', '}':'{'}
        stack = []
        for ch in s:
            if ch in pairs:                     # 右括号
                if not stack or stack[-1] != pairs[ch]:
                    return False
                stack.pop()
            else:                                # 左括号
                stack.append(ch)
        return not stack

sol20 = Solution_20()
cases = [("()",       True),
         ("()[]{}",   True),
         ("(]",       False),    # 顺序错
         ("([)]",     False),    # 交叉嵌套
         ("{[]}",     True),     # 合法嵌套
         ("",         True),     # 空串
         ("(",        False),    # 多余左括号
         (")",        False)]    # 多余右括号
for s, exp in cases:
    r = sol20.isValid(s)
    print(f"{'✅' if r==exp else '❌'}  {s!r:15} → {r} (期望 {exp})")

> **🎯 一句话总结**  
> 这道题的触发信号是 **字符串里「嵌套/最近才能配对」的成对符号 + 顺序敏感**，用的是 **栈匹配（LIFO + 哈希表存配对关系）** 模式，核心操作是 **左符号压栈、右符号弹栈比对，遍历完判栈是否为空**。

### 2. LC 150 — 逆波兰表达式求值

| 项目 | 内容 |
|------|------|
| **题型** | 栈求值（后缀表达式）|
| **时间复杂度** | O(n) |
| **空间复杂度** | O(n) |
| **数据范围** | 1 ≤ tokens.length ≤ 10⁴；运算符 ∈ `{+, -, *, /}`；除法向零截断 |

### 题目核心
**后缀表达式**（逆波兰）：运算符在两个操作数**之后**。如 `(2 + 1) * 3` 写成 `["2","1","+","3","*"]`。  
求值流程：遇数压栈、遇符号弹两个数算完再压回。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| 后缀表达式 / 逆波兰 | 栈求值 |
| 中缀转后缀 | Shunting Yard 算法（栈 + 运算符优先级）|
| 二叉树后序遍历的等价表达 | 同样能用栈复现 |

### 🧠 算法
```
stack = []
for token in tokens:
    if token in {'+','-','*','/'}:
        b = stack.pop()       # 注意：先弹的是右操作数
        a = stack.pop()       # 后弹的是左操作数
        stack.append(apply(a, op, b))
    else:
        stack.append(int(token))
return stack[-1]
```

### 🔑 三个关键点

**① 弹出顺序：先弹 b 后弹 a，计算 `a op b`**  
压栈顺序是 a 先 b 后，所以弹栈时先弹 b（右操作数），再弹 a（左操作数）。  
对加和乘无所谓（满足交换律），但**对减和除一定不能弄反**：`a - b ≠ b - a`。

**② 除法"向零截断"≠ Python 默认 `//`**  
Python 的 `//` 是**向下取整**（floor division）：`-7 // 2 == -4`  
题目要求**向零截断**（truncate toward zero）：`-7 / 2 == -3`（C/Java 风格）  
正确写法：`int(a / b)`（Python 的 `int()` 转换会向零截断）。

```python
# 反例：用 // 会出错
>>> -7 // 2
-4         # ❌（题目期望 -3）

# 正解：用 int(a/b)
>>> int(-7 / 2)
-3         # ✅
```

**③ 为什么不需要处理优先级？**  
后缀表达式的设计就是**让运算顺序在 token 序列里隐式编码**——读到运算符就立即用栈顶的两个数算，根本不需要知道"先加还是先乘"。这就是后缀表达式相比中缀表达式的优势。

In [ ]:
class Solution_150:
    def evalRPN(self, tokens: List[str]) -> int:
        stack = []
        ops = {'+', '-', '*', '/'}
        for token in tokens:
            if token in ops:
                b = stack.pop()                  # 先弹右操作数
                a = stack.pop()                  # 后弹左操作数
                if   token == '+': stack.append(a + b)
                elif token == '-': stack.append(a - b)
                elif token == '*': stack.append(a * b)
                else:              stack.append(int(a / b))   # 向零截断！
            else:
                stack.append(int(token))
        return stack[-1]

sol150 = Solution_150()
cases = [(["2","1","+","3","*"],                       9),  # (2+1)*3=9
         (["4","13","5","/","+"],                      6),  # 4 + (13/5) = 6
         (["10","6","9","3","+","-11","*","/","*","17","+","5","+"], 22),
         (["18"],                                      18),
         (["-7","2","/"],                              -3)] # 关键：向零截断
for tokens, exp in cases:
    r = sol150.evalRPN(tokens)
    print(f"{'✅' if r==exp else '❌'}  {tokens} → {r} (期望 {exp})")

> **🎯 一句话总结**  
> 这道题的触发信号是 **后缀表达式（逆波兰）求值**，用的是 **栈求值** 模式，核心操作是 **遇数压栈，遇运算符弹两个数（先弹的是右操作数）计算后再压回；除法用 `int(a/b)` 向零截断**。

---
## Part 2 — 数据结构设计

**核心思想**：在标准数据结构上**额外维护一个同步的辅助结构**，把"额外查询"摊到 O(1)。

### 3. LC 155 — 最小栈

| 项目 | 内容 |
|------|------|
| **题型** | 数据结构设计：双栈同步（辅助栈）|
| **时间复杂度** | push/pop/top/getMin 全部 **O(1)** |
| **空间复杂度** | O(n)（辅助栈与主栈同长）|

### 题目核心
设计支持 `push / pop / top / getMin` 的栈，**所有操作 O(1)**。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| 标准数据结构 + 额外查询要求 O(1) | 辅助结构同步维护查询所需信息 |
| 栈/队列上要求 O(1) 查"最小值/最大值/总和/众数..." | 双结构同步 |

### 🧠 算法（辅助栈思路）

主栈存所有元素；辅助栈 `min_stack` 在每次 push 时**同步压入"当前最小值"**：

```
push(x):
    main.append(x)
    min_stack.append( min(x, min_stack[-1]) )   # 与之前最小比较

pop():
    main.pop()
    min_stack.pop()                             # 同步弹出

getMin():
    return min_stack[-1]                        # 栈顶就是当前最小
```

**初始化技巧**：`min_stack = [float('inf')]`，让首次 push 时 `min(x, inf) == x`，避免空栈判断。

### 🔑 为什么这样能 O(1) 求最小？

辅助栈在**索引 i** 处存的是「主栈前 i+1 个元素的最小值」——这是一个**前缀最小值序列**。  
弹出主栈顶 = 弹出辅助栈顶，因为"前 i 个的最小值"与"前 i-1 个的最小值"一一对应，**同步维护就保证了正确性**。

### 替代方案：单栈 + 差值法（更省空间但难懂）

只用一个栈，存元素与当前最小值的**差值**；维护一个 `min` 变量。  
push/pop/getMin 都 O(1)，空间 O(n) 但常数更小。可读性差，面试中**优先讲双栈方案**。

### 与"队列实现栈"（LC 225）的对比

| | LC 155 最小栈 | LC 225 用队列实现栈 |
|--|--|--|
| 主体 | 标准栈 + 辅助栈 | 标准队列（or 两个）|
| 难点 | 同步维护额外信息 | 模拟另一种结构的操作 |
| 共同点 | 都是"数据结构设计"题型 | 都用**辅助结构**实现额外语义 |

In [ ]:
class MinStack:
    def __init__(self):
        self.stack = []
        # 哨兵：初始放 inf，避免每次 push 都判空
        self.min_stack = [float('inf')]

    def push(self, value: int) -> None:
        self.stack.append(value)
        # 关键：同步压入 "当前为止的最小值"
        self.min_stack.append(min(value, self.min_stack[-1]))

    def pop(self) -> None:
        self.stack.pop()
        self.min_stack.pop()                     # 同步弹

    def top(self) -> int:
        return self.stack[-1]

    def getMin(self) -> int:
        return self.min_stack[-1]


# 按题目示例验证
ms = MinStack()
ms.push(-2); ms.push(0); ms.push(-3)
assert ms.getMin() == -3, ms.getMin()           # 当前最小 = -3
ms.pop()                                          # 弹掉 -3
assert ms.top()    == 0,  ms.top()              # 栈顶剩 0
assert ms.getMin() == -2, ms.getMin()           # 当前最小 = -2
print("✅ 题目示例验证通过：[-2,0,-3] pop 后 top=0, min=-2")

# 边界测试：getMin 应正确处理多个相同最小值
ms2 = MinStack()
for v in [5, 3, 3, 7, 3]:
    ms2.push(v)
assert ms2.getMin() == 3
ms2.pop(); ms2.pop()                              # 弹出 3, 7
assert ms2.getMin() == 3                          # 仍有两个 3
ms2.pop()                                          # 弹出 3
assert ms2.getMin() == 3
ms2.pop()                                          # 弹出 3
assert ms2.getMin() == 5
print("✅ 边界测试：重复最小值连续弹出处理正确")

> **🎯 一句话总结**  
> 这道题的触发信号是 **栈结构 + 额外要求 O(1) 查最小值（或最大/众数等）**，用的是 **辅助栈（双栈同步）** 模式，核心操作是 **主栈正常 push/pop，辅助栈每次 push 同步存「当前最小值」，getMin 直接返回辅助栈顶**。

### 4. LC 225 — 用队列实现栈

| 项目 | 内容 |
|------|------|
| **题型** | 数据结构设计：队列模拟栈 |
| **时间复杂度** | push **O(n)**；pop/top/empty O(1)（单队列入队时翻转方案）|
| **空间复杂度** | O(n) |

### 题目核心
只用队列的标准操作（`push_back` / `pop_front` / `front` / `is_empty` / `size`）实现 **LIFO 栈**。

### ⚡ 触发条件
- 用 FIFO 结构模拟 LIFO 行为 → **入队时翻转**或**出队时翻转**
- 类似题：LC 232（用栈实现队列）

### 🧠 算法一：单队列 + push 时翻转 O(n) push、O(1) pop

```
push(x):
    q.append(x)
    # 把前面 n-1 个元素逐个弹出再压回 → 新元素就到队头了
    for _ in range(len(q) - 1):
        q.append(q.popleft())

pop()  → q.popleft()       # 现在队头就是栈顶
top()  → q[0]
empty()→ len(q) == 0
```

### 🧠 算法二：双队列方案 O(n) push、O(1) pop（对照）

```
push(x):
    q2.append(x)            # 新元素先进辅助队列
    while q1: q2.append(q1.popleft())  # 旧元素全转移到 q2 后面
    q1, q2 = q2, q1         # 角色互换
```

### 🧠 算法三：偷懒方案 — 直接用 list

> 截图代码就是这样写的：`self.stack = []`，直接调 list 的方法。  
> ⚠️ **这其实违背了题目"只用队列操作"的约束**，因为 list.pop() 是 O(1) 从尾部弹，等价于栈本身。  
> 在 Python 面试中，要么用 `collections.deque` 写出"入队时翻转"的真正队列方案，要么承认这是 trade-off。

### 单队列入队时翻转：为什么能行？

push 之后队列是 `[a, b, c, x]`，要把它变成"x 在队头"。  
循环 n-1 次"队头出来再加到队尾"：
```
[a, b, c, x]  →  [b, c, x, a]
[b, c, x, a]  →  [c, x, a, b]
[c, x, a, b]  →  [x, a, b, c]   ✅ 新元素到队头
```

### 与 LC 232（用栈实现队列）的镜像关系

| | LC 225 队列→栈 | LC 232 栈→队列 |
|--|--|--|
| 思路 | push 时翻转，pop 直接 popleft | 双栈：in 栈 push，out 栈 pop |
| 复杂度 | push O(n), pop O(1) | push O(1), pop 摊销 O(1) |

In [ ]:
class MyStack_Single:
    """单队列方案：push 时把队列旋转一圈，让新元素到队头"""
    def __init__(self):
        self.q = deque()

    def push(self, x: int) -> None:
        self.q.append(x)
        # 把之前的 n-1 个元素逐个搬到尾部 → 新元素就到队头
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())

    def pop(self) -> int:
        return self.q.popleft()

    def top(self) -> int:
        return self.q[0]

    def empty(self) -> bool:
        return len(self.q) == 0


class MyStack_Double:
    """双队列方案：q1 是主队列、q2 是辅助队列"""
    def __init__(self):
        self.q1 = deque()
        self.q2 = deque()

    def push(self, x: int) -> None:
        self.q2.append(x)                             # 新元素先进 q2
        while self.q1:                                # 把 q1 旧元素全部接到 q2 后面
            self.q2.append(self.q1.popleft())
        self.q1, self.q2 = self.q2, self.q1           # 角色互换

    def pop(self) -> int:    return self.q1.popleft()
    def top(self) -> int:    return self.q1[0]
    def empty(self) -> bool: return len(self.q1) == 0


# ── 测试两种方案 ──────────────────────────────────────
for name, Cls in [("单队列", MyStack_Single), ("双队列", MyStack_Double)]:
    s = Cls()
    s.push(1); s.push(2)
    assert s.top()   == 2
    assert s.pop()   == 2
    assert not s.empty()
    s.push(3); s.push(4)
    assert s.top()   == 4
    assert s.pop()   == 4
    assert s.pop()   == 3
    assert s.pop()   == 1
    assert s.empty()
    print(f"✅ {name}方案：push/pop/top/empty 全部正确")

> **🎯 一句话总结**  
> 这道题的触发信号是 **用 FIFO 队列模拟 LIFO 栈**，用的是 **队列模拟栈（push 时翻转 / 双队列轮换）** 模式，核心操作是 **单队列方案：push 后循环 n-1 次「popleft 再 append」让新元素旋转到队头；之后 pop/top 都是 O(1) 头部操作**。

---
## Part 3 — 单调栈

**核心思想**：维护一个栈，栈内元素**始终保持单调**（递增或递减）。每个元素**最多入栈一次、出栈一次** → 总操作 O(n)。

### 何时选哪种？

| 问题 | 用单调栈 | 单调方向 |
|------|---------|---------|
| 找**下一个/前一个更大**元素 | 单调**递减**栈（栈底→栈顶递减）| 来个更大的就把栈里所有小的弹出 |
| 找**下一个/前一个更小**元素 | 单调**递增**栈（栈底→栈顶递增）| 来个更小的就把栈里所有大的弹出 |
| 柱状图最大矩形 / 接雨水 | 单调**递增**栈 | 弹出时计算"以弹出元素为最矮"的区域 |
| 车队 / 落石问题 | 排序 + 栈或单次扫描 | 各题略有差异 |

### 通用模板
```python
stack = []                      # 栈里存索引或值
for i, x in enumerate(nums):
    while stack and 触发条件(stack[-1], x):
        idx = stack.pop()
        # ... 用 idx 和 i 计算答案 ...
    stack.append(i)             # 入栈
# 可能需要处理剩余元素
```

### 5. LC 739 — 每日温度（单调栈入门）

| 项目 | 内容 |
|------|------|
| **题型** | 单调递减栈（找下一个更大元素 NGE）|
| **时间复杂度** | O(n)（每个元素最多入栈出栈各一次）|
| **空间复杂度** | O(n) |
| **数据范围** | 1 ≤ n ≤ 10⁵，30 ≤ T[i] ≤ 100 |

### 题目核心
对每个 `i`，求"下一个比 T[i] 更高的温度"出现在几天之后；若没有则为 0。

### ⚡ 触发条件

| 信号 | 对应模式 |
|------|---------|
| "下一个更大/更高的..." | 单调**递减**栈（栈底→栈顶递减）|
| 答案为"**距离**"或"**索引差**" | 栈里存**索引**而非值 |
| 不存在时填默认值（如 0） | 答案数组初始化为该默认值 |

### 🧠 算法

栈内存"**还在等待更高温度**"的日子的**索引**，栈从底到顶**递减**（栈底是最早还在等的）：

```
stack = []                          # 单调递减栈，存索引
ans = [0] * n                       # 默认 0（没找到）
for i, t in enumerate(temperatures):
    while stack and temperatures[stack[-1]] < t:    # 当前温度比栈顶高
        prev = stack.pop()                            # 栈顶找到了 "下一个更高"
        ans[prev] = i - prev                          # 距离 = 索引差
    stack.append(i)                                   # 当前日子入栈等待
return ans
```

### 🔑 三个关键点

**① 为什么栈里存索引而不是温度？**  
因为答案要"几天之后"——是索引差。存索引才能算 `i - prev`。  
存值的话还得另外维护索引，麻烦。

**② while 而非 if**  
当前的高温可能让栈里**多个**之前的低温日子同时找到答案。如温度序列 `[1, 2, 3]`，到 3 时要同时给 1 和 2 设答案。

**③ 末尾剩余元素的处理**  
循环结束后栈里可能还有元素——它们就是**永远没等到更高温度**的日子，答案保持默认值 0。  
代码无需特殊处理，因为 ans 已初始化为 0。

### 与"找下一个**更小**元素"的对称写法

只要把 `while stack and T[stack[-1]] < t` 改成 `>` 就变成单调递增栈，找下一个更小元素。  
比较运算符的方向决定了栈的单调方向。

In [ ]:
class Solution_739:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        n = len(temperatures)
        ans = [0] * n                            # 默认 0：没找到更高温度
        stack = []                                # 单调递减栈：存索引
        for i, t in enumerate(temperatures):
            while stack and temperatures[stack[-1]] < t:
                prev = stack.pop()
                ans[prev] = i - prev              # 距离 = 索引差
            stack.append(i)
        return ans

sol739 = Solution_739()
cases = [([73,74,75,71,69,72,76,73], [1,1,4,2,1,1,0,0]),
         ([30,40,50,60],              [1,1,1,0]),    # 单调升
         ([30,60,90],                 [1,1,0]),
         ([90,80,70],                 [0,0,0]),       # 单调降，全 0
         ([55],                       [0])]
for t, exp in cases:
    r = sol739.dailyTemperatures(t)
    print(f"{'✅' if r==exp else '❌'}  {t} → {r}")

> **🎯 一句话总结**  
> 这道题的触发信号是 **求每个位置的「下一个更大/更高元素」的距离或索引**，用的是 **单调递减栈（栈底→栈顶递减，栈内存索引）** 模式，核心操作是 **遍历时栈内是「还在等更大者」的索引；遇到更大就 while 弹栈给答案赋距离 i - prev**。

### 6. LC 84 — 柱状图中最大的矩形 ⭐ Hard

| 项目 | 内容 |
|------|------|
| **题型** | 单调递增栈（找两侧第一个更小元素）|
| **时间复杂度** | O(n) |
| **空间复杂度** | O(n) |
| **数据范围** | 1 ≤ n ≤ 10⁵，0 ≤ heights[i] ≤ 10⁴ |
| **难度** | 🔴 困难（单调栈应用经典题）|

### 题目核心
柱状图中找一个**最大矩形**（必须由相邻柱子组成）。矩形面积 = **最矮柱子的高度 × 宽度**。

### ⚡ 触发条件

| 信号 | 对应技巧 |
|------|---------|
| 对每个位置 `i`，需要知道**左/右第一个比它更小的位置** | 单调递增栈 |
| "以 X 为最矮值的最大宽度" | 双向单调栈 |
| 接雨水（LC 42）、最大矩形（LC 85） | 同套路推广 |

### 🧠 算法（核心思想）

枚举每根柱子 `h[i]` 作为"矩形的最矮柱子"，问题变成：  
**「以 h[i] 为最矮、能向左右扩展多宽？」** —— 即找到左/右第一个比 h[i] **更小**的位置 L, R，宽度 = R - L - 1。

要找"两侧第一个更小元素" → **单调递增栈**：
- 栈内索引对应的高度从底到顶递增
- 当遇到一个**更小**的高度时，弹出栈顶，此时：
  - 弹出的元素 `h[t]` 就是中间这根"最矮柱"
  - 它**右侧第一个更小元素**就是当前 `i`（触发弹出的那个）
  - 它**左侧第一个更小元素**就是**弹出后**的新栈顶（如果空就用 -1）
  - 宽度 = `i - new_top - 1`，面积 = `h[t] * 宽度`

```python
heights.append(0)                   # 哨兵：末尾加 0 强制清空栈
stack = []                          # 单调递增栈
max_area = 0
for i, h in enumerate(heights):
    while stack and heights[stack[-1]] > h:
        t = stack.pop()
        left = stack[-1] if stack else -1
        width = i - left - 1
        max_area = max(max_area, heights[t] * width)
    stack.append(i)
return max_area
```

### 🔑 三个关键点

**① 末尾哨兵 0**  
若不加，栈里可能残留元素遍历结束后未结算。加一个 0（比所有 ≥0 的高度都小）能强制把所有剩余元素弹出。

**② 宽度公式 `width = i - left - 1`**  
- `i` = 当前位置（右边第一个 <h[t] 的索引）
- `left` = 弹出后的新栈顶（左边第一个 <h[t] 的索引）
- 中间能扩展的范围是 `(left, i)` 开区间，宽度是 `i - left - 1`

**③ 为什么是"两侧严格更小"而不是"小于等于"？**  
相等的情况下，重复计算不会丢答案（因为相同高度的柱子被分开计算时面积只会更小或相等），用 `>` 比 `>=` 更直观。

### 应用推广

- **LC 42 接雨水**：同样用单调栈，弹出时计算"以该位置为底"能接的雨水
- **LC 85 最大矩形**（二维）：对每一行用 LC 84 求最大矩形，逐行更新答案

In [ ]:
class Solution_84:
    def largestRectangleArea(self, heights: List[int]) -> int:
        heights = heights + [0]                 # 末尾哨兵：强制清空栈
        stack = []                               # 单调递增栈，存索引
        max_area = 0
        for i, h in enumerate(heights):
            while stack and heights[stack[-1]] > h:
                t = stack.pop()                   # 中间最矮的柱子索引
                # 弹出后的新栈顶即左侧第一个更小元素的位置；空栈代表 -1
                left = stack[-1] if stack else -1
                width = i - left - 1              # 开区间 (left, i) 的宽度
                max_area = max(max_area, heights[t] * width)
            stack.append(i)
        return max_area

sol84 = Solution_84()
cases = [([2,1,5,6,2,3],   10),   # 经典示例：5 和 6 组成 2×5=10
         ([2,4],             4),   # 单根 4
         ([1,1,1,1],         4),   # 4×1=4
         ([6,7,5,2,4,5,9,3], 16),  # 4×4=16（最后那段）
         ([2,1,2],           3)]   # 1×3=3
for hs, exp in cases:
    r = sol84.largestRectangleArea(hs)
    print(f"{'✅' if r==exp else '❌'}  {hs} → {r} (期望 {exp})")

> **🎯 一句话总结**  
> 这道题的触发信号是 **柱状图最大矩形面积（找以每根柱子为最矮的最大可扩展宽度）**，用的是 **单调递增栈 + 末尾哨兵 0** 模式，核心操作是 **遇到更矮高度就弹栈：弹出的索引 t 即「以 h[t] 为最矮」，宽度 = i - 新栈顶 - 1，结算面积**。

### 7. LC 853 — 车队（排序 + 栈式扫描）

| 项目 | 内容 |
|------|------|
| **题型** | 排序 + 栈/单次扫描（也可看作"追及关系判定"）|
| **时间复杂度** | O(n log n)（排序主导）|
| **空间复杂度** | O(n) |
| **数据范围** | 1 ≤ n ≤ 10⁵；0 ≤ position[i] < target ≤ 10⁶ |

### 题目核心
有 n 辆车开往同一目的地 `target`。给定每辆车的起始位置和速度。  
**慢车不会被快车超过**，被追上后两辆合并成一个"车队"，以慢车速度继续行驶。  
返回到达终点时形成的**车队数量**。

### 关键洞察 ⭐
**按起始位置从近到远（离终点越近排越前）排序后**，依次检查每辆车到终点所需时间：
- 若**当前车比前车慢**（time 更大）→ 自成一队
- 若**当前车比前车快**（time ≤ 前车）→ 会在终点前追上 → 合并进前面的车队

这等价于一个"单调递增的 time 栈"，但因为只关心栈顶（最近一个车队的 time），可以**直接用 lead 变量记录**而不真用栈。

### ⚡ 触发条件

| 信号 | 对应思路 |
|------|---------|
| 一维线性追及问题 / 落石问题 / 后浪推前浪 | 排序 + 栈式扫描 |
| 只关心"最近一个未被合并的实体" | 栈或单变量 lead |
| 比较"完成时间"而非"位置" | time = (target - position) / speed |

### 🧠 算法

```python
cars = sorted(zip(position, speed), reverse=True)    # 按 position 从近到远
times = [(target - p) / s for p, s in cars]

fleets = 0
lead = 0       # 当前车队头车到达时间
for t in times:
    if t > lead:                # 比前车慢 → 自成车队，更新 lead
        fleets += 1
        lead = t
    # else: 比前车快或一样 → 会追上 → 合并，不更新 lead
return fleets
```

### 🔑 三个关键点

**① 为什么是"按位置从近到远"排序？**  
离终点近的车 = 后面所有车追赶的目标。我们要按"被追"的顺序处理：先处理 leader，再依次看后面的车能不能追上。

**② `lead` 变量的语义**  
`lead` 记录当前最前面那个"独立车队"到达终点的时间。后面的车若 time 更大 → 永远追不上 → 自成一队，并成为新 leader；time 更小或相等 → 会被合并。

**③ 截图代码的写法（用 times.pop()）**  
```python
while len(times) > 1:
    lead = times.pop()       # 取最远（最早处理）那辆
    if lead < times[-1]:     # 比下一辆（更近）的 time 短 → 永远追不上
        ans += 1
    else:                    # 否则被合并：让下一辆继承 lead
        times[-1] = lead
return ans + bool(times)     # 处理最后一辆
```
等价的另一种循环写法，思路完全一样。

### 用栈来表达（显式版本）

```python
stack = []           # 栈内 time 严格递增
for t in times:      # 按位置从近到远
    if not stack or t > stack[-1]:
        stack.append(t)         # 自成车队
return len(stack)               # 车队数 = 栈深度
```

In [ ]:
class Solution_853:
    """简洁版：单变量 lead"""
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        # 按位置从近到远排序（离终点近的在前）
        cars = sorted(zip(position, speed), reverse=True)
        fleets = 0
        lead = 0.0                              # 当前最前面车队的到达时间
        for p, s in cars:
            t = (target - p) / s
            if t > lead:                        # 比前面那个车队慢 → 新车队
                fleets += 1
                lead = t
            # else: 追上前面车队，合并
        return fleets


class Solution_853_Stack:
    """显式栈版本"""
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        cars = sorted(zip(position, speed), reverse=True)
        stack = []                              # 栈内 time 严格递增
        for p, s in cars:
            t = (target - p) / s
            if not stack or t > stack[-1]:
                stack.append(t)                 # 自成车队
            # else: 被合并
        return len(stack)


for name, Sol in [("简洁版", Solution_853), ("栈版", Solution_853_Stack)]:
    sol = Sol()
    cases = [(12, [10,8,0,5,3], [2,4,1,1,3], 3),    # 题目示例
             (10, [3],           [3],        1),    # 单车
             (100,[0,2,4],       [4,2,1],    1),    # 全合并：4 追上 2，再追上 1
             (10, [0,4,2],       [2,1,3],    1),
             (10, [6,8],         [3,2],      2)]    # 6→10 用 4/3，8→10 用 1；前车更慢→分开
    for tgt, pos, spd, exp in cases:
        r = sol.carFleet(tgt, pos, spd)
        print(f"{'✅' if r==exp else '❌'}  [{name}] target={tgt}, pos={pos}, spd={spd} → {r} (期望 {exp})")
    print()

> **🎯 一句话总结**  
> 这道题的触发信号是 **一维线性追及合并问题（车队 / 落石 / 后浪推前浪）**，用的是 **按位置从近到远排序 + 栈式扫描（或单变量 lead）** 模式，核心操作是 **对排序后的车依次算到达时间 t：t>lead 则新车队 +1 并更新 lead，否则被合并**。

---
## Part 4 — 单调队列（滑动窗口最值）

### 单调栈 vs 单调队列

| | 单调栈 | 单调队列 |
|--|--|--|
| 数据结构 | `list`（只末尾操作）| `collections.deque`（双端都能操作）|
| 适用 | 找下一个更大/更小（无窗口大小限制）| 窗口内的最值（窗口大小固定/可变）|
| 弹元素的方向 | 只从末尾（栈顶）| **末尾**弹（破坏单调）+ **头部**弹（窗口过期）|

> 💡 单调队列 = 单调栈 + 头部出队功能。需要"窗口左边界过期就丢弃"时才用 deque。

### 8. LC 239 — 滑动窗口最大值 ⭐ Hard

| 项目 | 内容 |
|------|------|
| **题型** | 单调递减双端队列（单调队列）|
| **时间复杂度** | O(n)（每个元素最多入队出队各一次）|
| **空间复杂度** | O(k)（队列大小 ≤ 窗口大小 k）|
| **数据范围** | 1 ≤ n ≤ 10⁵，1 ≤ k ≤ n |
| **难度** | 🔴 困难 |

### 题目核心
长度 k 的滑动窗口从左到右扫过 nums，每一步输出窗口内的**最大值**。

### ⚡ 触发条件

| 信号 | 对应技巧 |
|------|---------|
| 滑动窗口 + **窗口内最值**（max / min）| 单调队列 |
| 朴素法 `for each window: max(window)` O(nk) 超时 | 用单调队列降到 O(n) |

### 🧠 算法（单调递减队列存索引）

队列内的**值**从队头到队尾**递减**（队头是当前窗口最大值的索引）。  
每来一个新元素 `nums[i]`：

```
1. 从队尾弹出所有 ≤ nums[i] 的元素（它们绝不会再成为最大，因为有了更大的 nums[i]）
2. 把 i 加入队尾
3. 如果队头索引 < i - k + 1（已超出窗口左边界），从队头弹出
4. 当 i >= k - 1（窗口完全形成）时，nums[deque[0]] 就是当前窗口最大值
```

```python
dq = deque()             # 存索引，对应值单调递减
result = []
for i, num in enumerate(nums):
    while dq and nums[dq[-1]] <= num:      # ① 尾部弹出"过时的小值"
        dq.pop()
    dq.append(i)                            # ② 入队
    if dq[0] == i - k:                      # ③ 头部弹出窗口外元素
        dq.popleft()
    if i >= k - 1:                          # ④ 窗口形成后，输出队头对应值
        result.append(nums[dq[0]])
return result
```

### 🔑 三个关键点

**① 为什么尾部弹出可以"放心丢"？**  
被弹出的元素 `nums[dq[-1]] <= num`：在 `nums[i]` 还在窗口里时，它**不可能成为最大值**（因为 i 更大且值更大或相等）；一旦 i 离开窗口，它早就已经离开了（i 比它更靠右）。所以这些"过时小值"永无翻身可能。

**② 为什么头部要按"索引"判断过期？**  
窗口左边界是 `i - k + 1`，所以队头若 `<` 此值就过期了。等价写法 `dq[0] == i - k` 也常见（同一意思）。

**③ 为什么队列里存索引而不是值？**  
需要判断"是否过期" → 必须知道元素的位置（索引）；存值的话无法判断窗口边界。

### 与"求最小值"的对称

只需把 `nums[dq[-1]] <= num` 改为 `>=`，单调方向翻转，就变成求滑动窗口最小值。

### 复杂度证明：均摊 O(n)

虽然有 while 循环看上去像 O(nk)，但每个元素**最多入队一次、出队一次** → 总操作次数 ≤ 2n。这是**均摊分析**的经典案例。

### 应用推广

- **LC 1438** 绝对差不超过限制的最长子数组：单调队列同时维护窗口 max 和 min
- **LC 862** 和至少为 K 的最短子数组：单调队列 + 前缀和

In [ ]:
class Solution_239:
    def maxSlidingWindow(self, nums: List[int], k: int) -> List[int]:
        dq = deque()                            # 存索引，对应值单调递减
        result = []
        for i, num in enumerate(nums):
            # ① 尾部弹出：当前 num 把之前比它小的"挤掉"
            while dq and nums[dq[-1]] <= num:
                dq.pop()
            # ② 入队
            dq.append(i)
            # ③ 头部弹出：超出窗口左边界
            if dq[0] == i - k:
                dq.popleft()
            # ④ 窗口形成（i >= k-1）后输出队头对应值
            if i >= k - 1:
                result.append(nums[dq[0]])
        return result

sol239 = Solution_239()
cases = [([1,3,-1,-3,5,3,6,7], 3, [3,3,5,5,6,7]),   # 题目示例
         ([1],                 1, [1]),
         ([1,-1],               1, [1,-1]),
         ([9,11],               2, [11]),
         ([4,-2],               2, [4]),
         ([1,3,1,2,0,5],        3, [3,3,2,5])]
for nums, k, exp in cases:
    r = sol239.maxSlidingWindow(nums, k)
    print(f"{'✅' if r==exp else '❌'}  nums={nums}, k={k} → {r}")

> **🎯 一句话总结**  
> 这道题的触发信号是 **固定/动态滑动窗口内的最大值或最小值**，用的是 **单调递减双端队列（deque 存索引）** 模式，核心操作是 **新元素入队前 while 弹尾部所有 ≤ 自己的旧元素；队头超出窗口左边界时 popleft；窗口形成后取队头索引对应值**。

---
## 📋 总结：栈与队列全模板

### 八题对照表

| 题号 | 题型 | 时间 | 空间 | 核心技巧 | 关键易错点 |
|------|------|------|------|---------|-----------|
| 20   | 基础栈：括号匹配 | O(n) | O(n) | 哈希表 + 栈 | 哈希表用「右→左」省一次判断；末尾要检查栈空 |
| 150  | 栈求值（后缀表达式）| O(n) | O(n) | 数字压栈、运算符弹两个 | 弹出顺序 b 在前 a 在后；除法用 `int(a/b)` 向零截断 |
| 155  | 数据结构设计：辅助栈 | O(1)/op | O(n) | 双栈同步存「当前最小」| 用 `[inf]` 哨兵省判空；pop 时**同步弹**两个栈 |
| 225  | 队列模拟栈 | push O(n) | O(n) | 入队时循环旋转让新元素到队头 | 用 `deque` 而非 `list` 才符合题目语义 |
| 739  | 单调递减栈（NGE）| O(n) | O(n) | 栈内存索引，弹出时 `ans[t]=i-t` | while 不是 if（一次可解决多个） |
| 853  | 排序 + 栈式扫描 | O(n log n) | O(n) | 按位置降序后单变量 lead 跟踪 | 排序方向（从近到远）；time 比较而非 position |
| 84   | 单调递增栈（双侧 NSE）| O(n) | O(n) | 末尾哨兵 0，弹出时算 `h[t]*(i-left-1)` | left 是**弹出后**的栈顶（不存在用 -1） |
| 239  | 单调递减双端队列 | O(n) | O(k) | 尾部弹小、头部弹过期 | 队头按**索引**判过期；存索引而非值 |

---

### 🔁 栈/队列核心模板

```python
# ── 基础栈：括号匹配（LC 20）──────────────────────────────
pairs = {')':'(', ']':'[', '}':'{'}
stack = []
for ch in s:
    if ch in pairs:                                # 右括号
        if not stack or stack[-1] != pairs[ch]: return False
        stack.pop()
    else: stack.append(ch)                          # 左括号
return not stack

# ── 后缀表达式求值（LC 150）────────────────────────────────
stack = []
for token in tokens:
    if token in OPS:
        b = stack.pop(); a = stack.pop()
        stack.append(apply(a, op, b))               # 注意 a op b 的顺序
    else: stack.append(int(token))
return stack[-1]

# ── 辅助栈：常数时间查询额外信息（LC 155 最小栈）─────────────
class MinStack:
    def __init__(self): self.s=[]; self.m=[float('inf')]
    def push(self,x): self.s.append(x); self.m.append(min(x,self.m[-1]))
    def pop(self):    self.s.pop(); self.m.pop()
    def getMin(self): return self.m[-1]

# ── 单调递减栈（找下一个更大元素，LC 739）─────────────────
stack=[]; ans=[0]*n
for i, t in enumerate(arr):
    while stack and arr[stack[-1]] < t:
        prev = stack.pop()
        ans[prev] = i - prev
    stack.append(i)

# ── 单调递增栈（双侧 NSE，LC 84 柱状图最大矩形）─────────────
heights.append(0)                                   # 末尾哨兵
stack = []; max_area = 0
for i, h in enumerate(heights):
    while stack and heights[stack[-1]] > h:
        t = stack.pop()
        left = stack[-1] if stack else -1            # 弹出后的栈顶
        max_area = max(max_area, heights[t] * (i - left - 1))
    stack.append(i)

# ── 排序 + 栈式扫描（LC 853 车队）──────────────────────────
cars = sorted(zip(position, speed), reverse=True)   # 按位置从近到远
fleets, lead = 0, 0.0
for p, s in cars:
    t = (target - p) / s
    if t > lead: fleets += 1; lead = t              # 自成车队
    # else: 被合并

# ── 单调递减双端队列（LC 239 滑动窗口最大值）──────────────
dq = deque(); result = []
for i, num in enumerate(nums):
    while dq and nums[dq[-1]] <= num: dq.pop()      # 尾部弹小
    dq.append(i)
    if dq[0] == i - k: dq.popleft()                  # 头部弹过期
    if i >= k - 1: result.append(nums[dq[0]])

# ── 队列模拟栈（LC 225）─────────────────────────────────────
class MyStack:
    def __init__(self): self.q = deque()
    def push(self, x):
        self.q.append(x)
        for _ in range(len(self.q)-1):
            self.q.append(self.q.popleft())          # 循环旋转
    def pop(self):  return self.q.popleft()
    def top(self):  return self.q[0]
```

---

### 🗺️ 栈/队列题型识别速查（增强版）

```
嵌套/匹配/最近原则               → 栈                                (LC 20, LC 32)
后缀表达式求值 / 中缀转后缀        → 栈 + 运算符判断                    (LC 150, LC 224)
数据结构 + O(1) 查最值/总和        → 辅助栈（双栈同步）                 (LC 155, LC 716)
用 A 结构模拟 B 结构（栈/队列互换）→ 数据结构设计                      (LC 225, LC 232)
下一个/前一个更大/更小元素         → 单调栈                            (LC 496, LC 503, LC 739, LC 901)
柱状图最大矩形 / 接雨水            → 单调（递增）栈 + 双侧 NSE          (LC 42, LC 84, LC 85)
车队 / 股票收盘 / 落石问题         → 排序 + 栈式扫描                    (LC 853, LC 901)
滑动窗口的最值 / 区间最值          → 单调队列（deque）                  (LC 239, LC 1438, LC 862)
```

> ⚠️ **遇到 "下一个 / 前一个" → 单调栈；遇到 "窗口内" → 单调队列。**  
> 这是栈/队列题最快速的分类判断。

---
*复习笔记 · 2026-06 · By Claude*